In [165]:
from google import genai
from dotenv import load_dotenv
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

In [166]:
load_dotenv()

client = genai.Client()

In [167]:
class ReviewResponseState(TypedDict):

    review: str
    sentiment_class: Literal['positive', 'negative']
    diagnosis: dict
    review_response: str

In [168]:
class get_sentiment_class_structured_output(BaseModel):

    sentiment_class: Literal['positive', 'negative'] = Field(description= "Sentiment of the review")

In [169]:
def get_sentiment_class(State: ReviewResponseState) -> ReviewResponseState:

    review = State['review']

    response = client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = f"Categorize this review: '{review}' into positive or negative class",
        config = {
            "response_mime_type": 'application/json',
            "response_schema": get_sentiment_class_structured_output
        }
    )

    return {
        "sentiment_class": response.parsed.sentiment_class
    }

In [170]:
def get_positive_response(State: ReviewResponseState) -> ReviewResponseState:

    review = State['review']

    response = client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = f"This is a positive review given by a customer for a particular electronic gadget, review: '{review}', Generate back a response which I should give to the customer. Output should only contain the response, nothing else."
    )

    return {
        "review_response": response.text
    }

In [171]:
class negative_review_diagnosis_structured_output(BaseModel):

    issue_type: str = Field(description= "The issue type of the review")
    tone: Literal['anger', 'frustation', 'moderate', 'disappointement', 'sarcasm'] = Field(description= "The tone of the review")
    urgency: Literal['very_urgent', 'urgent', 'moderate', 'low'] = Field(description= "The urgency of the review")

In [172]:
def run_diagnosis(State: ReviewResponseState) -> ReviewResponseState:

    review = State['review']

    response = client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = f"Identify the issue_type, tone and urgency of the review: '{review}' clearly",
        config = {
            "response_mime_type": "application/json",
            "response_schema": negative_review_diagnosis_structured_output
        }
    )

    diagnosis_dict = {
        'issue_type': response.parsed.issue_type,
        'tone': response.parsed.tone,
        'urgency': response.parsed.urgency
    }

    return {
        'diagnosis': diagnosis_dict
    }

In [173]:
def get_negative_response(State: ReviewResponseState) -> ReviewResponseState:

    diagnosis = State['diagnosis']

    response = client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = f"This is a negative review given by a customer for a particular electronic gadget, review: '{review}', Generate back a response which I should give to the customer keeping in mind the issue_type: {diagnosis['issue_type']}, tone: {diagnosis['tone']} and urgency: {diagnosis['urgency']} of the review. Output should only contain the response, nothing else."
    )

    return {
        'review_response': response.text
    }

In [174]:
def check_condition(State: ReviewResponseState) -> Literal['get_positive_response', 'run_diagnosis']:

    if State['sentiment_class'] == "positive":
        return "get_positive_response"
    else:
        return "run_diagnosis"

In [175]:
graph = StateGraph(ReviewResponseState)

graph.add_node('get_sentiment_class', get_sentiment_class)
graph.add_node('get_positive_response', get_positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('get_negative_response', get_negative_response)

graph.add_edge(START, 'get_sentiment_class')
graph.add_conditional_edges('get_sentiment_class', check_condition)
graph.add_edge('get_positive_response', END)
graph.add_edge('run_diagnosis', 'get_negative_response')
graph.add_edge('get_negative_response', END)

workflow = graph.compile()

In [176]:
review = "CMF by Nothing Buds 2a (₹1,999) turned out to be a disappointing budget purchase. The sound feels thin and lacks bass despite the 12.4mm bio-fibre driver claims, and the 42dB 'hybrid' ANC barely cancels out background noise in real-world use. Battery life falls well short of the advertised 35.5 hours, draining much faster with regular use. Build quality feels cheap, the fit is uncomfortable after long wear, and the water resistance seems more marketing than functional. Not worth the price."

initial_state = {
    "review": review
}

final_state = workflow.invoke(initial_state)

final_state

{'review': "CMF by Nothing Buds 2a (₹1,999) turned out to be a disappointing budget purchase. The sound feels thin and lacks bass despite the 12.4mm bio-fibre driver claims, and the 42dB 'hybrid' ANC barely cancels out background noise in real-world use. Battery life falls well short of the advertised 35.5 hours, draining much faster with regular use. Build quality feels cheap, the fit is uncomfortable after long wear, and the water resistance seems more marketing than functional. Not worth the price.",
 'sentiment_class': 'negative',
 'diagnosis': {'issue_type': 'product quality and performance',
  'tone': 'disappointement',
  'urgency': 'moderate'},
 'review_response': 'We are genuinely sorry to hear about your experience with the CMF Buds 2a, as this certainly does not reflect the standard of quality and performance we aim to deliver. We completely understand your frustration regarding the sound profile, ANC effectiveness, battery life, and overall build quality. Your feedback is ex